# Diagnosing Inference Problems

## Learning Objectives
By completing this notebook, you will:
1. Master comprehensive MCMC diagnostic workflow
2. Interpret divergences, R-hat, ESS, and E-BFMI
3. Use trace plots and posterior predictive checks effectively
4. Troubleshoot common inference failures systematically
5. Apply diagnostics to commodity forecasting models

## Prerequisites
- NUTS algorithm understanding (Module 6.3)
- Experience with PyMC sampling
- Basic knowledge of MCMC theory

## Estimated Time: 65 minutes

---

In [ ]:
learning_objectives(["Master comprehensive MCMC diagnostic workflow", "Interpret divergences, R-hat, ESS, and E-BFMI", "Use trace plots and posterior predictive checks effectively", "Troubleshoot common inference failures systematically", "Apply diagnostics to commodity forecasting models", "NUTS algorithm understanding (Module 6.3)"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import diagnostic tools
# Key Concept: ArviZ is the gold standard for Bayesian diagnostics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"PyMC: {pm.__version__}")
print(f"ArviZ: {az.__version__}")

## 1. The Diagnostic Workflow

A systematic approach to checking MCMC results:

```
1. Visual inspection (trace plots)
   ↓
2. Convergence metrics (R-hat, ESS)
   ↓
3. Sampling quality (divergences, E-BFMI, treedepth)
   ↓
4. Model adequacy (posterior predictive checks)
   ↓
5. Decision: trust results or debug
```

Let's build this workflow systematically.

## 2. Trace Plots: Visual Inspection

Trace plots show the sampled values over iterations. Good traces look like "hairy caterpillars."

In [ ]:
# Purpose: Generate good and bad traces for comparison
# Key Concept: Visual patterns reveal sampling issues

# Good model: simple regression
np.random.seed(123)
X_good = np.random.randn(100, 2)
y_good = X_good @ [2, -1] + np.random.randn(100) * 0.5

with pm.Model() as good_model:
    beta = pm.Normal('beta', 0, 10, shape=2)
    sigma = pm.Exponential('sigma', 1)
    mu = pm.math.dot(X_good, beta)
    obs = pm.Normal('obs', mu, sigma, observed=y_good)
    
    trace_good = pm.sample(500, tune=500, chains=4, random_seed=42, 
                           return_inferencedata=True, progressbar=False)

# Bad model: highly correlated parameters
with pm.Model() as bad_model:
    x = pm.Normal('x', 0, 1)
    y = pm.Normal('y', x * 10, 0.1)  # Very tight constraint
    
    trace_bad = pm.sample(500, tune=500, chains=4, random_seed=42,
                          return_inferencedata=True, progressbar=False)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Good traces
az.plot_trace(trace_good, var_names=['beta'], compact=False, axes=axes[0, :])
axes[0, 0].set_title('Good: Hairy Caterpillar')

# Bad traces
az.plot_trace(trace_bad, var_names=['x'], axes=axes[1, :])
axes[1, 0].set_title('Bad: Slow Mixing')

plt.tight_layout()
plt.show()

print("""
Good trace characteristics:
✅ All chains overlap
✅ No trends or drift
✅ Rapid mixing (high frequency oscillation)
✅ Stationary distribution

Bad trace characteristics:
❌ Chains separated
❌ Slow exploration
❌ Autocorrelation visible
""")

## 3. R-hat: Convergence Diagnostic

**R-hat** (Gelman-Rubin statistic) compares between-chain and within-chain variance.

- **R-hat ≈ 1.00**: Chains have converged
- **R-hat > 1.01**: Chains haven't mixed; run longer
- **R-hat > 1.1**: Serious convergence issues

**Rule**: All parameters should have R-hat < 1.01

In [ ]:
# Purpose: Demonstrate R-hat interpretation
# Key Concept: R-hat measures chain agreement

# Compute R-hat
rhat_good = az.rhat(trace_good)
rhat_bad = az.rhat(trace_bad)

print("="*60)
print("R-HAT COMPARISON")
print("="*60)

print("\nGood model:")
print(rhat_good)
print(f"Max R-hat: {max(rhat_good.max().values()):.4f}")

print("\nBad model:")
print(rhat_bad)
print(f"Max R-hat: {max(rhat_bad.max().values()):.4f}")

# Visualize with forest plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

az.plot_forest(trace_good, var_names=['beta'], r_hat=True, ax=axes[0])
axes[0].set_title('Good Model: R-hat < 1.01')

az.plot_forest(trace_bad, var_names=['x'], r_hat=True, ax=axes[1])
axes[1].set_title('Bad Model: High R-hat')

plt.tight_layout()
plt.show()

print("\n💡 R-hat shown as markers on forest plots")

## 4. ESS: Effective Sample Size

**ESS** measures independent information in correlated samples.

- **ESS bulk**: For posterior mean estimates
- **ESS tail**: For extreme quantiles (important for risk)

**Rule**: Want ESS > 400 per chain (>1,600 for 4 chains)

In [ ]:
# Purpose: Compute and interpret ESS
# Key Concept: ESS tells us how many "independent" samples we have

ess_good = az.ess(trace_good)
ess_bad = az.ess(trace_bad)

print("="*60)
print("EFFECTIVE SAMPLE SIZE")
print("="*60)

print("\nGood model:")
print(ess_good)

print("\nBad model:")
print(ess_bad)

# Summary stats
summary_good = az.summary(trace_good, var_names=['beta'])
summary_bad = az.summary(trace_bad, var_names=['x'])

print("\nGood model summary:")
print(summary_good[['mean', 'sd', 'ess_bulk', 'ess_tail', 'r_hat']])

print("\nBad model summary:")
print(summary_bad[['mean', 'sd', 'ess_bulk', 'ess_tail', 'r_hat']])

print("""
\n💡 ESS Interpretation:
- ESS = # draws: Perfect (no autocorrelation)
- ESS = 50% of draws: Acceptable
- ESS < 400: Need more samples or better mixing
""")

## 5. Divergences: The Red Flag

**Divergences** occur when the leapfrog integrator becomes unstable.

**Causes:**
- Posterior geometry is too complex
- Step size is too large
- Model has pathological regions

**Solutions:**
1. Increase `target_accept` to 0.95 or 0.99
2. Reparameterize (especially hierarchical models)
3. Check for parameter constraints

In [ ]:
# Purpose: Create model with divergences and diagnose
# Key Concept: Divergences indicate biased posterior samples

# Eight schools (classic divergence example)
J = 8
y = np.array([28., 8., -3., 7., -1., 1., 18., 12.])
sigma = np.array([15., 10., 16., 11., 9., 11., 10., 18.])

# Centered parameterization (causes divergences)
with pm.Model() as centered:
    mu = pm.Normal('mu', mu=0, sigma=10)
    tau = pm.HalfCauchy('tau', beta=10)
    
    # Centered
    theta = pm.Normal('theta', mu=mu, sigma=tau, shape=J)
    
    obs = pm.Normal('obs', mu=theta, sigma=sigma, observed=y)
    
    trace_divergent = pm.sample(1000, tune=1000, chains=4, random_seed=42,
                                return_inferencedata=True, progressbar=False)

# Check divergences
n_div = trace_divergent.sample_stats.diverging.sum().values
print(f"\nNumber of divergences: {n_div}")

if n_div > 0:
    print("⚠️ Divergences detected! Posterior samples may be biased.")
    
# Visualize divergences
az.plot_pair(
    trace_divergent,
    var_names=['mu', 'tau'],
    divergences=True,
    figsize=(10, 8)
)
plt.suptitle('Divergences (red) concentrate in funnel neck', y=1.02)
plt.tight_layout()
plt.show()

print("""
💡 Divergence patterns:
- Concentrate in specific region → reparameterize
- Scattered everywhere → check model specification
- Only a few → try increasing target_accept
""")

In [ ]:
# Purpose: Fix divergences with non-centered parameterization
# Key Concept: Reparameterization often resolves divergences

# Non-centered parameterization
with pm.Model() as noncentered:
    mu = pm.Normal('mu', mu=0, sigma=10)
    tau = pm.HalfCauchy('tau', beta=10)
    
    # Non-centered
    theta_raw = pm.Normal('theta_raw', mu=0, sigma=1, shape=J)
    theta = pm.Deterministic('theta', mu + tau * theta_raw)
    
    obs = pm.Normal('obs', mu=theta, sigma=sigma, observed=y)
    
    trace_fixed = pm.sample(1000, tune=1000, chains=4, random_seed=42,
                           return_inferencedata=True, progressbar=False)

n_div_fixed = trace_fixed.sample_stats.diverging.sum().values

print("="*60)
print("BEFORE vs AFTER Reparameterization")
print("="*60)
print(f"Centered divergences: {n_div}")
print(f"Non-centered divergences: {n_div_fixed}")

if n_div_fixed == 0:
    print("\n✅ Problem solved!")

## 6. E-BFMI: Energy Diagnostics

**E-BFMI** (Energy Bayesian Fraction of Missing Information) checks if the sampler explores the entire distribution.

- **E-BFMI > 0.3**: Good
- **E-BFMI < 0.3**: Poor exploration, possible bias

Low E-BFMI often indicates model misspecification or need for reparameterization.

In [ ]:
# Purpose: Compute E-BFMI
# Key Concept: Low E-BFMI indicates exploration issues

ebfmi_centered = az.bfmi(trace_divergent)
ebfmi_fixed = az.bfmi(trace_fixed)

print("="*60)
print("E-BFMI DIAGNOSTIC")
print("="*60)

print(f"\nCentered model: {ebfmi_centered.values}")
print(f"Mean: {ebfmi_centered.values.mean():.3f}")

if ebfmi_centered.values.mean() < 0.3:
    print("⚠️ Low E-BFMI indicates exploration issues")

print(f"\nNon-centered model: {ebfmi_fixed.values}")
print(f"Mean: {ebfmi_fixed.values.mean():.3f}")

if ebfmi_fixed.values.mean() > 0.3:
    print("✅ E-BFMI is healthy")

# Energy plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

az.plot_energy(trace_divergent, ax=axes[0])
axes[0].set_title(f'Centered (E-BFMI={ebfmi_centered.values.mean():.3f})')

az.plot_energy(trace_fixed, ax=axes[1])
axes[1].set_title(f'Non-centered (E-BFMI={ebfmi_fixed.values.mean():.3f})')

plt.tight_layout()
plt.show()

print("""
💡 Energy plot interpretation:
- Distributions should overlap well
- Large separation → poor exploration
""")

## 7. Posterior Predictive Checks

**PPC** checks if the model can generate data similar to observed data.

**Method:**
1. Sample from posterior: $\theta^{(s)} \sim p(\theta|y)$
2. Generate replicated data: $y^{rep,(s)} \sim p(y|\theta^{(s)})$
3. Compare $y^{rep}$ to $y$

In [ ]:
# Purpose: Perform posterior predictive check
# Key Concept: PPC validates model adequacy

# Generate commodity price data
np.random.seed(789)
n_days = 500
true_returns = stats.t.rvs(df=5, loc=0.001, scale=0.02, size=n_days)  # Heavy tails

# Fit Normal model (misspecified)
with pm.Model() as normal_model:
    mu = pm.Normal('mu', 0, 0.1)
    sigma = pm.Exponential('sigma', 50)
    obs = pm.Normal('obs', mu, sigma, observed=true_returns)
    
    trace_normal = pm.sample(1000, tune=1000, chains=2, random_seed=42,
                            return_inferencedata=True, progressbar=False)
    
    # Generate posterior predictive samples
    ppc_normal = pm.sample_posterior_predictive(trace_normal, random_seed=42)

# Fit Student-t model (correct)
with pm.Model() as t_model:
    mu = pm.Normal('mu', 0, 0.1)
    sigma = pm.Exponential('sigma', 50)
    nu = pm.Gamma('nu', alpha=2, beta=0.1)
    obs = pm.StudentT('obs', nu=nu, mu=mu, sigma=sigma, observed=true_returns)
    
    trace_t = pm.sample(1000, tune=1000, chains=2, random_seed=42,
                       return_inferencedata=True, progressbar=False)
    
    ppc_t = pm.sample_posterior_predictive(trace_t, random_seed=42)

# Compare
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Normal model PPC
az.plot_ppc(trace_normal, ax=axes[0, 0])
axes[0, 0].set_title('Normal Model: PPC')

# Normal model: extreme values
ppc_samples_normal = ppc_normal.posterior_predictive['obs'].values.reshape(-1)
axes[0, 1].hist(true_returns, bins=50, alpha=0.5, label='Observed', density=True)
axes[0, 1].hist(ppc_samples_normal, bins=50, alpha=0.5, label='Predicted', density=True)
axes[0, 1].set_xlim(-0.15, 0.15)
axes[0, 1].set_title('Normal Model: Tails')
axes[0, 1].legend()

# Student-t model PPC
az.plot_ppc(trace_t, ax=axes[1, 0])
axes[1, 0].set_title('Student-t Model: PPC')

# Student-t model: extreme values
ppc_samples_t = ppc_t.posterior_predictive['obs'].values.reshape(-1)
axes[1, 1].hist(true_returns, bins=50, alpha=0.5, label='Observed', density=True)
axes[1, 1].hist(ppc_samples_t, bins=50, alpha=0.5, label='Predicted', density=True)
axes[1, 1].set_xlim(-0.15, 0.15)
axes[1, 1].set_title('Student-t Model: Tails')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("""
💡 PPC Interpretation:
✅ Student-t captures heavy tails
❌ Normal underestimates extreme values
→ Student-t is better model for this data
""")

## 8. Complete Diagnostic Workflow Function

In [ ]:
# Purpose: Comprehensive diagnostic function
# Key Concept: Automate the diagnostic workflow

def diagnose_trace(trace, var_names=None, verbose=True):
    """
    Comprehensive MCMC diagnostic function.
    
    Parameters
    ----------
    trace : InferenceData
        Posterior samples from PyMC
    var_names : list, optional
        Variables to check
    verbose : bool
        Print detailed output
    
    Returns
    -------
    dict : Diagnostic results
    """
    results = {}
    issues = []
    
    # 1. Divergences
    n_div = trace.sample_stats.diverging.sum().values
    results['n_divergences'] = n_div
    
    if n_div > 0:
        issues.append(f"⚠️ {n_div} divergences detected")
    
    # 2. R-hat
    rhat = az.rhat(trace, var_names=var_names)
    max_rhat = max([rhat[v].max().values for v in rhat.data_vars])
    results['max_rhat'] = max_rhat
    
    if max_rhat > 1.01:
        issues.append(f"⚠️ Max R-hat = {max_rhat:.4f} > 1.01")
    
    # 3. ESS
    ess = az.ess(trace, var_names=var_names)
    min_ess_bulk = min([ess[v].min().values for v in ess.data_vars])
    results['min_ess_bulk'] = min_ess_bulk
    
    if min_ess_bulk < 400:
        issues.append(f"⚠️ Min ESS = {min_ess_bulk:.0f} < 400")
    
    # 4. E-BFMI
    ebfmi = az.bfmi(trace)
    mean_ebfmi = ebfmi.values.mean()
    results['mean_ebfmi'] = mean_ebfmi
    
    if mean_ebfmi < 0.3:
        issues.append(f"⚠️ E-BFMI = {mean_ebfmi:.3f} < 0.3")
    
    # 5. Max treedepth
    if hasattr(trace.sample_stats, 'max_energy_error'):
        max_td = trace.sample_stats.reached_max_treedepth.sum().values
        results['max_treedepth_reached'] = max_td
        
        if max_td > 0:
            issues.append(f"⚠️ Reached max treedepth {max_td} times")
    
    results['issues'] = issues
    results['passed'] = len(issues) == 0
    
    if verbose:
        print("="*60)
        print("DIAGNOSTIC REPORT")
        print("="*60)
        print(f"\nDivergences: {n_div}")
        print(f"Max R-hat: {max_rhat:.4f}")
        print(f"Min ESS (bulk): {min_ess_bulk:.0f}")
        print(f"Mean E-BFMI: {mean_ebfmi:.3f}")
        
        if len(issues) == 0:
            print("\n✅ All diagnostics passed!")
        else:
            print("\nIssues found:")
            for issue in issues:
                print(f"  {issue}")
            print("\n💡 Recommendations:")
            if n_div > 0:
                print("  - Increase target_accept to 0.95")
                print("  - Consider reparameterization")
            if max_rhat > 1.01:
                print("  - Run longer chains")
            if min_ess_bulk < 400:
                print("  - Draw more samples")
            if mean_ebfmi < 0.3:
                print("  - Check model specification")
                print("  - Try reparameterization")
    
    return results

# Test the function
print("Testing on good model:")
diagnose_trace(trace_good)

print("\n" + "="*60)
print("\nTesting on problematic model:")
diagnose_trace(trace_divergent)

## 9. Exercises

### Exercise 6.5.1: Diagnose and Fix

**Task:** Run diagnostics on the model below and fix any issues.

In [ ]:
# Exercise: Hierarchical model with issues
np.random.seed(111)
n_groups = 10
n_per_group = 20
group_means = np.random.randn(n_groups) * 3
exercise_data = np.concatenate([
    np.random.randn(n_per_group) * 0.5 + group_means[i]
    for i in range(n_groups)
])
group_idx = np.repeat(np.arange(n_groups), n_per_group)

# Centered parameterization (will have issues)
with pm.Model() as exercise_model:
    mu_global = pm.Normal('mu_global', 0, 10)
    sigma_global = pm.Exponential('sigma_global', 1)
    mu_group = pm.Normal('mu_group', mu_global, sigma_global, shape=n_groups)
    sigma = pm.Exponential('sigma', 2)
    obs = pm.Normal('obs', mu_group[group_idx], sigma, observed=exercise_data)
    
    trace_exercise = pm.sample(500, tune=500, chains=2, random_seed=42,
                              return_inferencedata=True, progressbar=False)

# YOUR CODE HERE
# ---------------
# 1. Run diagnose_trace() on trace_exercise
# 2. Identify issues
# 3. Fix the model using non-centered parameterization
# 4. Verify the fix works

# Step 1: Diagnose
# diag_results = diagnose_trace(trace_exercise)

# Step 2-3: Fix
# with pm.Model() as fixed_model:
#     ...
#     trace_fixed_ex = pm.sample(...)

# Step 4: Verify
# diagnose_trace(trace_fixed_ex)

trace_fixed_ex = None  # Replace with your fixed trace

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_6_5_1():
    assert trace_fixed_ex is not None, "Create trace_fixed_ex"
    
    results = diagnose_trace(trace_fixed_ex, verbose=False)
    
    assert results['n_divergences'] < 5, "Too many divergences"
    assert results['max_rhat'] < 1.01, "R-hat too high"
    assert results['passed'], "Some diagnostics failed"
    
    print("✅ Exercise 6.5.1 passed!")
    print(f"   Fixed model has {results['n_divergences']} divergences")
    print(f"   Max R-hat: {results['max_rhat']:.4f}")

# Uncomment to test:
# test_exercise_6_5_1()

### Exercise 6.5.2: Posterior Predictive Check

**Task:** Perform a PPC and determine if the model is adequate.

In [ ]:
# Exercise: Check model adequacy
np.random.seed(222)
# Generate data with changepoint
n_before = 100
n_after = 100
ppc_data = np.concatenate([
    np.random.randn(n_before) * 1.0 + 0,
    np.random.randn(n_after) * 2.0 + 3
])

# Fit simple model (no changepoint)
with pm.Model() as simple_model:
    mu = pm.Normal('mu', 0, 10)
    sigma = pm.Exponential('sigma', 1)
    obs = pm.Normal('obs', mu, sigma, observed=ppc_data)
    
    trace_simple = pm.sample(1000, tune=1000, chains=2, random_seed=42,
                            return_inferencedata=True, progressbar=False)

# YOUR CODE HERE
# ---------------
# 1. Generate posterior predictive samples
# 2. Plot PPC
# 3. Determine if model is adequate

# with simple_model:
#     ppc = pm.sample_posterior_predictive(trace_simple)

is_adequate = None  # True or False
reason = ""  # Why is it adequate or not?

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_6_5_2():
    assert is_adequate is not None, "Set is_adequate"
    assert isinstance(is_adequate, bool), "is_adequate should be boolean"
    assert reason != "", "Provide reason"
    
    # Model is NOT adequate due to changepoint
    assert is_adequate == False, "Model misses changepoint structure"
    assert 'change' in reason.lower() or 'regime' in reason.lower(), \
        "Reason should mention changepoint/regime"
    
    print("✅ Exercise 6.5.2 passed!")
    print(f"   Your assessment: {'Adequate' if is_adequate else 'Not adequate'}")
    print(f"   Reason: {reason}")

# Uncomment to test:
# test_exercise_6_5_2()

### Exercise 6.5.3: Complete Diagnostic Workflow

**Task:** Apply the complete workflow to a commodity model.

In [ ]:
# Exercise: Commodity volatility model
np.random.seed(333)
n_days = 250
true_vol = np.zeros(n_days)
true_vol[0] = 0
for t in range(1, n_days):
    true_vol[t] = 0.9 * true_vol[t-1] + np.random.randn() * 0.3

commodity_returns = np.random.randn(n_days) * np.exp(true_vol / 2)

# YOUR CODE HERE
# ---------------
# 1. Build stochastic volatility model
# 2. Sample with NUTS
# 3. Run complete diagnostics
# 4. Perform PPC
# 5. Summarize findings

# with pm.Model() as sv_model:
#     ...
#     trace_sv = pm.sample(...)

# Diagnostics
# diag = diagnose_trace(trace_sv)

# PPC
# with sv_model:
#     ppc_sv = pm.sample_posterior_predictive(trace_sv)

summary_report = ""  # Your summary

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_6_5_3():
    assert summary_report != "", "Provide summary report"
    assert len(summary_report) > 100, "Summary should be detailed"
    
    # Check key concepts mentioned
    report_lower = summary_report.lower()
    checks = [
        'divergence' in report_lower or 'rhat' in report_lower,
        'ess' in report_lower or 'effective' in report_lower,
        'ppc' in report_lower or 'predictive' in report_lower
    ]
    
    assert sum(checks) >= 2, "Summary should mention key diagnostics"
    
    print("✅ Exercise 6.5.3 passed!")
    print(f"\nYour summary ({len(summary_report)} chars):")
    print(summary_report[:200] + "...")

# Uncomment to test:
# test_exercise_6_5_3()

---

## Summary

### Key Takeaways

1. **Systematic workflow**: Always follow the diagnostic checklist
2. **Trace plots**: Visual inspection reveals mixing and convergence
3. **R-hat < 1.01**: Essential convergence criterion
4. **ESS > 400**: Minimum effective sample size per chain
5. **Divergences = red flag**: Indicate biased samples
6. **E-BFMI > 0.3**: Checks exploration quality
7. **Reparameterization**: Non-centered often fixes hierarchical issues
8. **PPC validation**: Model must generate realistic data
9. **Automate diagnostics**: Use functions like diagnose_trace()
10. **Never trust without checking**: Even "successful" sampling needs verification

### Diagnostic Checklist

✅ Visual inspection (trace plots)  
✅ R-hat < 1.01 for all parameters  
✅ ESS > 400 per chain  
✅ No divergences (or very few with high target_accept)  
✅ E-BFMI > 0.3  
✅ No max treedepth warnings  
✅ Posterior predictive check passes  
✅ Results make scientific sense  

### Common Issues & Solutions

| Issue | Likely Cause | Solution |
|-------|--------------|----------|
| Divergences | Funnel geometry | Non-centered parameterization |
| High R-hat | Poor mixing | Longer chains, reparameterization |
| Low ESS | High autocorrelation | More samples, better parameterization |
| Low E-BFMI | Model misspecification | Check model, try reparameterization |
| Max treedepth | Complex posterior | Increase max_treedepth to 12-15 |
| PPC fails | Wrong model | Revise model structure |

### What's Next

- **Next module**: Regime Switching Models for commodities
- **Advanced diagnostics**: LOO-CV, WAIC for model comparison
- **Application**: Production-ready diagnostic pipelines

### Additional Resources

- Vehtari et al. (2021): "Rank-Normalization, Folding, and Localization: An Improved R-hat"
- Betancourt (2018): "Diagnosing Biased Inference with Divergences"
- ArviZ documentation: https://arviz-devs.github.io/arviz/
- PyMC case studies: https://www.pymc.io/projects/examples/en/latest/